In [3]:
import json
import pandas as pd
import numpy as np

In [4]:
candidates = []

with open("candidates.jsonl", "r") as f:
    for line in f:
        candidates.append(json.loads(line))

print(len(candidates))

100000


In [5]:
def extract_skills(candidate):
    return [
        skill["name"]
        for skill in candidate["skills"]
    ]

In [6]:
extract_skills(candidates[0])

['Tailwind',
 'NLP',
 'Image Classification',
 'Fine-tuning LLMs',
 'Weights & Biases',
 'Speech Recognition',
 'Photoshop',
 'TTS',
 'LoRA',
 'Apache Beam',
 'AWS',
 'Flask',
 'BentoML',
 'Milvus',
 'GANs',
 'Statistical Modeling',
 'GCP']

In [7]:
jd = """
Looking for NLP Engineer with
Python, NLP, LLM,
Milvus, Flask
"""

In [8]:
required_skills = [
    "Python",
    "NLP",
    "LLM",
    "Milvus",
    "Flask"
]

In [9]:
def skill_match_score(candidate,required_skills):
    candidate_skills = [
        s["name"].lower()
        for s in candidate["skills"]
    ]

    matches = sum(
        1
        for skill in required_skills
        if skill.lower() in candidate_skills
    )

    return matches / len(required_skills)

In [10]:
skill_match_score(candidates[0],required_skills)

0.6

In [11]:
def experience_score(candidate):
    
    years = candidate["profile"]["years_of_experience"]

    return min(years / 10, 1.0)

In [12]:
def recruiter_signal_score(candidate):

    signals = candidate["redrob_signals"]

    score = (
        signals["recruiter_response_rate"] +
        signals["interview_completion_rate"] +
        signals["offer_acceptance_rate"]
    ) / 3

    return score

In [13]:
def profile_quality_score(candidate):

    signals = candidate["redrob_signals"]

    score = (
        signals["profile_completeness_score"] / 100
    )

    return score

In [14]:
def skill_strength_score(candidate):

    prof_map = {
        "beginner": 1,
        "intermediate": 2,
        "advanced": 3
    }

    total = 0

    for skill in candidate["skills"]:

        proficiency = prof_map.get(
            skill["proficiency"].lower(),
            1
        )

        total += proficiency

    return total / (len(candidate["skills"]) * 3)

In [15]:
def final_candidate_score(candidate,required_skills):

    skill_score = skill_match_score(candidate,required_skills)

    exp_score = experience_score(candidate)

    recruiter_score = recruiter_signal_score(candidate)

    profile_score = profile_quality_score(candidate)

    strength_score = skill_strength_score(candidate)

    final = (0.50 * skill_score + 0.20 * recruiter_score + 0.15 * exp_score + 0.10 * strength_score + 0.05 * profile_score)

    return final

In [17]:
final_candidate_score(candidates[0],required_skills)

0.6301264705882352

In [18]:
scores = []

for candidate in candidates[:1000]:

    score = final_candidate_score(candidate,required_skills)

    scores.append((candidate["candidate_id"],score))

In [19]:
sorted(scores,key=lambda x: x[1],reverse=True)[:10]

[('CAND_0000001', 0.6301264705882352),
 ('CAND_0000422', 0.5672409090909092),
 ('CAND_0000127', 0.47374285714285724),
 ('CAND_0000247', 0.4675333333333333),
 ('CAND_0000167', 0.44789814814814816),
 ('CAND_0000760', 0.4458987179487179),
 ('CAND_0000037', 0.4404666666666667),
 ('CAND_0000803', 0.4393777777777778),
 ('CAND_0000390', 0.4357925925925926),
 ('CAND_0000136', 0.4334944444444445)]

In [20]:
all_skills = set()

for candidate in candidates[:5000]:

    for skill in candidate["skills"]:
        all_skills.add(skill["name"])

print(len(all_skills))

119


In [21]:
list(all_skills)[:20]

['ETL',
 'MLOps',
 'Marketing',
 'LLMs',
 'Reinforcement Learning',
 'Agile',
 'FAISS',
 'Prompt Engineering',
 'Project Management',
 'Snowflake',
 'RAG',
 'REST APIs',
 'SEO',
 'Weaviate',
 'Image Classification',
 'Tailwind',
 'Kafka',
 'TypeScript',
 'Computer Vision',
 'Hugging Face Transformers']

In [26]:
def extract_skills_from_jd(jd, skill_vocabulary):

    jd_lower = jd.lower()

    found_skills = set()

    for skill in skill_vocabulary:

        if skill.lower() in jd_lower:
            found_skills.add(skill)

    for phrase, skill in skill_aliases.items():

        if phrase.lower() in jd_lower:
            found_skills.add(skill)

    return list(found_skills)

In [133]:
jd = """
Looking for NLP Engineer with
Python, Flask,
LLM deployment experience
and vector database knowledge.
"""

extract_skills_from_jd(jd,all_skills)

['Flask', 'LLMs', 'Milvus', 'NLP', 'Python']

In [28]:
all_skills = set()

for candidate in candidates[:5000]:
    for skill in candidate["skills"]:
        all_skills.add(skill["name"])

print(len(all_skills))
print(list(all_skills)[:30])

119
['ETL', 'MLOps', 'Marketing', 'LLMs', 'Reinforcement Learning', 'Agile', 'FAISS', 'Prompt Engineering', 'Project Management', 'Snowflake', 'RAG', 'REST APIs', 'SEO', 'Weaviate', 'Image Classification', 'Tailwind', 'Kafka', 'TypeScript', 'Computer Vision', 'Hugging Face Transformers', 'CSS', 'GANs', 'Milvus', 'gRPC', 'dbt', 'GraphQL', 'Photoshop', 'Kubernetes', 'BM25', 'LoRA']


In [29]:
skill_aliases = {
    "Large Language Models": "LLMs",
    "LLM": "LLMs",
    "Vector Database": "Milvus",
    "Vector Databases": "Milvus",
    "Retrieval Augmented Generation": "RAG",
    "Retrieval-Augmented Generation": "RAG",
    "Computer Vision": "Computer Vision",
    "Natural Language Processing": "NLP",
    "Backend APIs": "REST APIs"
}


In [132]:
jd = """
Looking for an AI Engineer with experience in

Large Language Models,
Vector Databases,
Retrieval Augmented Generation,
Python APIs.
"""

extract_skills_from_jd(jd,all_skills)

['Milvus', 'RAG', 'Python', 'LLMs']

In [31]:
def get_candidate_skills(candidate):

    skills = []

    for skill in candidate["skills"]:
        skills.append(skill["name"])

    return skills

In [32]:
get_candidate_skills(
    candidates[0]
)[:10]

['Tailwind',
 'NLP',
 'Image Classification',
 'Fine-tuning LLMs',
 'Weights & Biases',
 'Speech Recognition',
 'Photoshop',
 'TTS',
 'LoRA',
 'Apache Beam']

In [33]:
def skill_match_score(candidate_skills,required_skills):

    matches = 0

    for skill in required_skills:

        if skill in candidate_skills:
            matches += 1

    return matches / len(required_skills)

In [79]:
def rank_candidates_from_jd(jd,candidates,top_n=10):

    required_skills = extract_skills_from_jd(jd,all_skills)

    required_exp = extract_required_experience(jd)

    scores = []

    for candidate in candidates:

        score = final_candidate_score(candidate,required_skills,required_exp)

        scores.append((candidate["candidate_id"],score))

    return sorted(scores,key=lambda x: x[1],reverse=True)[:top_n]

In [100]:
rank_candidates_from_jd(
"""
Looking for AI Engineer with

5+ years experience,
LLMs,
RAG,
Python
""",
candidates[:1000]
)

[('CAND_0000739', 0.7679054545454547),
 ('CAND_0000201', 0.7673354545454546),
 ('CAND_0000664', 0.7351954545454547),
 ('CAND_0000399', 0.7169454545454547),
 ('CAND_0000803', 0.6969036363636365),
 ('CAND_0000722', 0.6854536363636364),
 ('CAND_0000699', 0.6849518181818183),
 ('CAND_0000083', 0.6780018181818183),
 ('CAND_0000828', 0.6705736363636364),
 ('CAND_0000581', 0.6666554545454546)]

In [81]:
def candidate_quality_score(candidate):

    signals = candidate["redrob_signals"]

    profile_score = (signals["profile_completeness_score"]/ 100)

    response_score = (signals["recruiter_response_rate"])

    interview_score = (signals["interview_completion_rate"])

    github_score = (signals["github_activity_score"]/ 100)

    return (0.35 * profile_score + 0.25 * response_score + 0.25 * interview_score + 0.15 * github_score)

In [82]:
candidate_quality_score(
    candidates[0]
)

0.5804500000000001

In [83]:
role_skill_map = {

    "genai engineer": ["LLMs","RAG","Prompt Engineering","Milvus","FAISS","LoRA"],

    "ml engineer": ["Python","MLOps","PyTorch"],

    "computer vision engineer": ["Computer Vision","Image Classification","GANs"],

    "data engineer": ["Kafka","Snowflake","ETL","dbt"]
}

In [84]:
def expand_jd_skills(jd, skills):

    jd_lower = jd.lower()

    expanded = set(skills)

    for role, role_skills in role_skill_map.items():

        if role in jd_lower:

            expanded.update(role_skills)

    return list(expanded)

In [134]:
import re

def extract_required_experience(jd):
    
    jd = jd.lower()

    match = re.search(r'(\d+)\+?\s*(?:years|yrs)',jd)

    if match:
        return int(match.group(1))

    return None

In [86]:
def experience_score(candidate_exp, required_exp):

    if required_exp is None:
        return 0.5

    ratio = candidate_exp / required_exp

    if ratio >= 1:
        return 1.0

    elif ratio >= 0.8:
        return 0.8

    elif ratio >= 0.5:
        return 0.5

    else:
        return 0.2

In [87]:
def proficiency_to_score(level):

    level = level.lower()

    mapping = {"advanced":1.0,"intermediate":0.7,"beginner":0.4}

    return mapping.get(level,0.5)

In [88]:
def skill_match_score_with_proficiency(candidate_skills,required_skills):

    if len(required_skills) == 0:
        return 0

    score = 0

    for skill in candidate_skills:

        skill_name = skill["name"].lower()

        for req_skill in required_skills:

            if req_skill.lower() in skill_name:

                score += proficiency_to_score(
                    skill["proficiency"]
                )

                break

    return score / len(required_skills)

In [89]:
candidates[0]["career_history"]

[{'company': 'Mindtree',
  'title': 'Backend Engineer',
  'start_date': '2024-03-08',
  'end_date': None,
  'duration_months': 27,
  'is_current': True,
  'industry': 'IT Services',
  'company_size': '10001+',
  'description': 'Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure.'},
 {'company': 'Dunder Mifflin',
  'title': 'Analytics Engineer',
  'start_date': '2019-07-03',
  'end_date': '2024-01-08',
  'duration_months': 55,
  'is_current': False,
  'industry': 'Paper Products',
  'company_size': '201-500',
  'description': 'Built and maintained data pipelines on Apache Airflow processing ~500GB of da

In [90]:
required_skills = [
    "LLMs",
    "RAG",
    "Python"
]

skill_match_score_with_proficiency(
    candidates[0]["skills"],
    required_skills
)

0.3333333333333333

In [91]:
candidates[0]["skills"][:5]

[{'name': 'Tailwind',
  'proficiency': 'intermediate',
  'endorsements': 3,
  'duration_months': 13},
 {'name': 'NLP',
  'proficiency': 'advanced',
  'endorsements': 37,
  'duration_months': 26},
 {'name': 'Image Classification',
  'proficiency': 'advanced',
  'endorsements': 7,
  'duration_months': 40},
 {'name': 'Fine-tuning LLMs',
  'proficiency': 'advanced',
  'endorsements': 21,
  'duration_months': 36},
 {'name': 'Weights & Biases',
  'proficiency': 'intermediate',
  'endorsements': 13,
  'duration_months': 30}]

In [92]:
required_skills

['LLMs', 'RAG', 'Python']

In [93]:
def career_relevance_score(candidate):

    text = ""

    for job in candidate["career_history"]:
        text += job["title"] + " "
        text += job["description"] + " "

    text = text.lower()

    matches = 0

    for keyword in AI_KEYWORDS:

        if keyword in text:
            matches += 1

    return matches / len(AI_KEYWORDS)

In [94]:
career_relevance_score(candidates[0])

0.18181818181818182

In [95]:
for job in candidates[0]["career_history"]:
    print(job["title"])
    print("-"*50)
    print(job["description"])
    print("\n")

Backend Engineer
--------------------------------------------------
Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-registry integration, the watermark/state management approach, and the deduplication logic for late-arriving events. Worked closely with the data science team to make sure feature pipelines aligned with what their models needed. Most of my career has been data engineering, with some adjacent ML exposure.


Analytics Engineer
--------------------------------------------------
Built and maintained data pipelines on Apache Airflow processing ~500GB of daily transactional data across 12 source systems. Worked extensively with Spark (PySpark) for batch processing and dbt for the transformation/modeling layer in our Snowflake warehouse. Owned the on-call rotation for data quality issues — wrote most of the data quality checks that detect schema drift and unusual volume changes. The pipeline

In [96]:
AI_KEYWORDS = ["machine learning","ml","ai","deep learning","llm","nlp","computer vision","rag","transformer","neural network","python"]

In [97]:
text = ""

for job in candidates[0]["career_history"]:
    text += job["title"] + " "
    text += job["description"] + " "

text = text.lower()

for keyword in AI_KEYWORDS:
    if keyword in text:
        print(keyword)

ml
ai


In [160]:
def final_candidate_score(jd,candidate,required_skills,required_exp):

    semantic_score = semantic_similarity_cached(jd,candidate)

    skill_score = skill_match_score_with_proficiency(candidate["skills"],required_skills)

    exp_score = experience_score(candidate["profile"]["years_of_experience"],required_exp)

    quality_score = candidate_quality_score(candidate)

    career_score = career_relevance_score(candidate)

    role_score = role_alignment_score(candidate)

    llm_score = llm_bonus(candidate)

    return (0.35 * semantic_score + 0.20 * skill_score + 0.10 * exp_score + 0.15 * quality_score + 0.10 * career_score + 0.05 * role_score + 0.05 * llm_score)

In [116]:
def build_candidate_profile(candidate):

    profile = ""

    profile += candidate["profile"]["headline"] + " "

    profile += candidate["profile"]["summary"] + " "

    for skill in candidate["skills"]:
        profile += skill["name"] + " "

    for job in candidate["career_history"]:
        profile += job["title"] + " "
        profile += job["description"] + " "

    return profile

In [117]:
print(
    build_candidate_profile(
        candidates[0]
    )[:1000]
)

Backend Engineer | SQL, Spark, Cloud Software / data professional with 6.9 years of experience building data pipelines, backend systems, and analytics infrastructure. I'm a backend/data hybrid — Spark, Airflow, SQL warehouses are home territory; I'm building competence on the ML side. My toolkit is solid on the data engineering side — Python, SQL, Spark, Airflow, warehouse design — and I've completed a couple of self-directed ML projects (Kaggle competitions, side projects fine-tuning small models). Interested in transitioning toward more AI/ML-focused work, ideally at a company where I can leverage my existing data-infra skills while learning modern ML practice. Tailwind NLP Image Classification Fine-tuning LLMs Weights & Biases Speech Recognition Photoshop TTS LoRA Apache Beam AWS Flask BentoML Milvus GANs Statistical Modeling GCP Backend Engineer Implemented streaming data pipelines on Kafka and Spark Streaming for a real-time user-activity processing platform. Designed the schema-r

In [104]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\bhite\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhite\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [118]:
embedding = model.encode(
    build_candidate_profile(candidates[0])
)

print(type(embedding))
print(len(embedding))

<class 'numpy.ndarray'>
384


In [119]:
from sklearn.metrics.pairwise import cosine_similarity

In [120]:
def semantic_similarity(jd_text, candidate):

    jd_embedding = model.encode(jd_text)

    candidate_embedding = model.encode(build_candidate_profile(candidate))

    similarity = cosine_similarity([jd_embedding],[candidate_embedding])[0][0]

    return similarity

In [121]:
semantic_similarity(
"""
Looking for AI Engineer with

LLMs
RAG
Python
""",
candidates[0]
)

np.float32(0.3098702)

In [168]:
candidate_embeddings = {}

for candidate in candidates[:1000]:

    candidate_embeddings[
        candidate["candidate_id"]
    ] = model.encode(
        
        build_candidate_profile(candidate)
    )

In [169]:
len(candidate_embeddings)

1000

In [124]:
for i in [0, 100, 500]:
    print(
        i,
        semantic_similarity(
        """
        Looking for AI Engineer with

        LLMs
        RAG
        Python
        """,
        candidates[i]
        )
    )

0 0.3098702
100 0.29983634
500 0.38148344


In [125]:
from sklearn.metrics.pairwise import cosine_similarity

def semantic_similarity_cached(jd_text,candidate):

    jd_embedding = model.encode(jd_text)

    candidate_embedding = candidate_embeddings[candidate["candidate_id"]]

    similarity = cosine_similarity([jd_embedding],[candidate_embedding])[0][0]

    return similarity

In [126]:
semantic_similarity_cached(
"""
Looking for AI Engineer with

LLMs
RAG
Python
""",
candidates[500]
)

np.float32(0.38148344)

In [127]:
def final_candidate_score_semantic(jd,candidate,required_skills,required_exp):

    semantic_score = semantic_similarity_cached(jd,candidate)

    skill_score = skill_match_score_with_proficiency(candidate["skills"],required_skills)

    quality_score = candidate_quality_score(candidate)

    career_score = career_relevance_score(candidate,required_skills,required_exp)

    return (0.45 * semantic_score + 0.30 * skill_score + 0.15 * quality_score + 0.10 * career_score)

In [136]:
def rank_candidates_semantic(jd,candidates,top_n=10):

    required_skills = extract_skills_from_jd(jd,all_skills)

    required_exp = extract_required_experience(jd)

    scores = []

    for candidate in candidates:

        score = final_candidate_score(jd,candidate,required_skills,required_exp)

        scores.append((candidate["candidate_id"],score))

    return sorted(scores,key=lambda x: x[1],reverse=True)[:top_n]

In [137]:
results = rank_candidates_semantic(
"""
Looking for AI Engineer with

5+ years experience
LLMs
RAG
Python
Vector Databases
Fine Tuning
""",
candidates[:1000]
)

results

[('CAND_0000739', np.float32(0.6438265)),
 ('CAND_0000201', np.float32(0.6138714)),
 ('CAND_0000422', np.float32(0.6059727)),
 ('CAND_0000399', np.float32(0.5877579)),
 ('CAND_0000581', np.float32(0.57865775)),
 ('CAND_0000664', np.float32(0.57689834)),
 ('CAND_0000699', np.float32(0.571403)),
 ('CAND_0000083', np.float32(0.57097507)),
 ('CAND_0000722', np.float32(0.5698594)),
 ('CAND_0000828', np.float32(0.56908786))]

In [138]:
top_id = results[0][0]

for c in candidates[:1000]:
    if c["candidate_id"] == top_id:
        top_candidate = c
        break

In [139]:
top_candidate["profile"]["headline"]

'Civil Engineer | AI enthusiast | Building with LLMs'

In [140]:
top_candidate["profile"]["summary"]

"Civil Engineer with 11.1+ years of experience driving outcomes in my domain. I have built strong functional expertise in the typical responsibilities of the role, including team management, stakeholder communication, and project delivery. Recently I've been excited about how AI and GenAI tools can augment my work. I've been taking online courses on RAG and vector databases, experimenting with LangChain and the OpenAI API for side projects, and exploring how LLMs can streamline workflows in my current function. Open to roles that combine my existing domain experience with emerging AI technologies — I think the most interesting opportunities are at this intersection. Looking for positions where I can contribute both my functional expertise and grow my AI capabilities."

In [141]:
top_candidate["skills"][:10]

[{'name': 'Illustrator',
  'proficiency': 'intermediate',
  'endorsements': 5,
  'duration_months': 15},
 {'name': 'Redis',
  'proficiency': 'intermediate',
  'endorsements': 4,
  'duration_months': 22},
 {'name': 'Data Pipelines',
  'proficiency': 'beginner',
  'endorsements': 8,
  'duration_months': 4},
 {'name': 'OpenCV',
  'proficiency': 'intermediate',
  'endorsements': 5,
  'duration_months': 20},
 {'name': 'Vue.js',
  'proficiency': 'intermediate',
  'endorsements': 15,
  'duration_months': 26},
 {'name': 'FastAPI',
  'proficiency': 'intermediate',
  'endorsements': 8,
  'duration_months': 10},
 {'name': 'Hugging Face Transformers',
  'proficiency': 'advanced',
  'endorsements': 3,
  'duration_months': 9},
 {'name': 'Semantic Search',
  'proficiency': 'intermediate',
  'endorsements': 4,
  'duration_months': 17},
 {'name': 'Prompt Engineering',
  'proficiency': 'advanced',
  'endorsements': 2,
  'duration_months': 6},
 {'name': 'Vector Search',
  'proficiency': 'advanced',
  'en

In [142]:
top_candidate["profile"]["years_of_experience"]

11.1

In [143]:
semantic_similarity_cached(
"""
Looking for AI Engineer with

5+ years experience
LLMs
RAG
Python
Vector Databases
Fine Tuning
""",
top_candidate
)

np.float32(0.6256306)

In [153]:
ai_roles = [
    "ai engineer",
    "ml engineer",
    "machine learning engineer",
    "data scientist",
    "nlp engineer",
    "llm engineer",
    "ai research engineer",
    "genai engineer",
    "generative ai engineer",
    "mlops engineer",
    "machine learning scientist",
    "ai scientist",
    "data engineer",
    "backend engineer"
]
def role_alignment_score(candidate):

    text = candidate["profile"]["headline"].lower()

    for job in candidate["career_history"]:
        text += " " + job["title"].lower()

    ai_roles = ["ai engineer","ml engineer","machine learning engineer","data scientist","nlp engineer","llm engineer","ai research engineer"]

    matches = 0

    for role in ai_roles:
        if role in text:
            matches += 1

    return min(matches / 3, 1.0)

In [154]:
for i in [0, 100, 500]:
    print(
        candidates[i]["profile"]["headline"],
        role_alignment_score(candidates[i])
    )

Backend Engineer | SQL, Spark, Cloud 0.0
Customer Support | Driving business outcomes 0.0
DevOps Engineer | Full-stack development 0.0


In [155]:
count = 0

for candidate in candidates[:1000]:

    score = role_alignment_score(candidate)

    if score > 0:
        print(
            score,
            candidate["profile"]["headline"]
        )
        count += 1

        if count >= 20:
            break

0.6666666666666666 Recommendation Systems Engineer | Search, Ranking & Retrieval
0.3333333333333333 AI Specialist | Building ML-powered solutions
0.3333333333333333 ML Engineer | Data Science & ML enthusiast
0.3333333333333333 ML Engineer | 5.8 yrs in analytics & ML
0.6666666666666666 AI Research Engineer | Data Science & ML enthusiast
1.0 Data Scientist | Building ML-powered solutions
0.6666666666666666 AI Research Engineer | Building ML-powered solutions
0.3333333333333333 ML Engineer | Data Science & ML enthusiast


In [156]:
for candidate in candidates[:1000]:

    score = role_alignment_score(candidate)

    if score >= 0.33:
        print(
            score,
            candidate["profile"]["headline"]
        )
        break

0.6666666666666666 Recommendation Systems Engineer | Search, Ranking & Retrieval


In [157]:
results = rank_candidates_semantic(
"""
Looking for AI Engineer with

5+ years experience
LLMs
RAG
Python
Vector Databases
Fine Tuning
""",
candidates[:1000]
)

for cid, score in results:
    candidate = next(
        c for c in candidates[:1000]
        if c["candidate_id"] == cid
    )

    print(
        round(float(score),4),
        candidate["profile"]["headline"]
    )

0.6133 AI Research Engineer | Data Science & ML enthusiast
0.5813 Civil Engineer | AI enthusiast | Building with LLMs
0.5579 Marketing Manager | AI enthusiast | Building with LLMs
0.5292 Business Analyst | AI enthusiast | Building with LLMs
0.5271 Accountant | Generative AI explorer
0.5204 Graphic Designer | Exploring AI & GenAI applications
0.5154 Business Analyst | Exploring AI & GenAI applications
0.515 Graphic Designer | Generative AI explorer
0.5143 HR Manager | AI enthusiast | Building with LLMs
0.5113 Project Manager | Generative AI explorer


In [158]:
top_id = results[0][0]

top_candidate = next(
    c for c in candidates[:1000]
    if c["candidate_id"] == top_id
)

print(top_candidate["profile"]["headline"])
print()
print(top_candidate["profile"]["summary"])

AI Research Engineer | Data Science & ML enthusiast

Data scientist / ML engineer with 6.3 years of experience in applied machine learning. Worked across predictive modeling, NLP, analytics, and lightweight deployment workflows. My current role is split between dashboarding/analytics and shipping production ML models. I'm strongest at the modeling and analysis side; comfortable with Python, scikit-learn, pandas, and standard MLOps tooling, but I'm still building depth on the engineering and infra side of production ML. Looking for a role where I can step up to more end-to-end ownership of ML systems, not just modeling.


In [159]:
LLM_SKILLS = [
    "llms",
    "rag",
    "vector databases",
    "vector search",
    "langchain",
    "fine tuning",
    "hugging face",
    "lora",
    "openai"
]
def llm_bonus(candidate):

    text = build_candidate_profile(candidate).lower()

    matches = 0

    for skill in LLM_SKILLS:
        if skill in text:
            matches += 1

    return matches / len(LLM_SKILLS)

In [161]:
def explain_candidate(candidate, jd, required_skills, required_exp):

    candidate_skills = [
        skill["name"].lower()
        for skill in candidate["skills"]
    ]

    matched_skills = []
    missing_skills = []

    for skill in required_skills:
        if skill.lower() in candidate_skills:
            matched_skills.append(skill)
        else:
            missing_skills.append(skill)

    years_exp = candidate["profile"]["years_of_experience"]

    semantic_score = semantic_similarity_cached(jd, candidate)
    skill_score = skill_match_score_with_proficiency(
        candidate["skills"],
        required_skills
    )

    exp_score = experience_score(
        years_exp,
        required_exp
    )

    llm_score = llm_bonus(candidate)

    return {
        "candidate_id": candidate["candidate_id"],
        "headline": candidate["profile"]["headline"],
        "experience": years_exp,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "semantic_score": round(float(semantic_score), 3),
        "skill_score": round(float(skill_score), 3),
        "experience_score": round(float(exp_score), 3),
        "llm_score": round(float(llm_score), 3)
    }

In [162]:
results = rank_candidates_semantic(
    jd,
    candidates[:1000]
)

In [163]:
top_id = results[0][0]

top_candidate = next(
    c for c in candidates[:1000]
    if c["candidate_id"] == top_id
)

In [164]:
explanation = explain_candidate(
    top_candidate,
    jd,
    required_skills,
    required_exp
)

explanation

{'candidate_id': 'CAND_0000422',
 'headline': 'AI Research Engineer | Data Science & ML enthusiast',
 'experience': 6.3,
 'matched_skills': ['Python'],
 'missing_skills': ['LLMs', 'RAG'],
 'semantic_score': 0.58,
 'skill_score': 0.333,
 'experience_score': 1.0,
 'llm_score': 0.111}

In [165]:
def print_explanation(explanation):

    print("=" * 60)

    print("Candidate:")
    print(explanation["headline"])

    print()

    print("Experience:")
    print(explanation["experience"], "years")

    print()

    print("Matched Skills:")
    for skill in explanation["matched_skills"]:
        print("✓", skill)

    print()

    print("Missing Skills:")
    for skill in explanation["missing_skills"]:
        print("✗", skill)

    print()

    print("Scores:")
    print("Semantic:", explanation["semantic_score"])
    print("Skill:", explanation["skill_score"])
    print("Experience:", explanation["experience_score"])
    print("LLM:", explanation["llm_score"])

In [166]:
print_explanation(explanation)

Candidate:
AI Research Engineer | Data Science & ML enthusiast

Experience:
6.3 years

Matched Skills:
✓ Python

Missing Skills:
✗ LLMs
✗ RAG

Scores:
Semantic: 0.58
Skill: 0.333
Experience: 1.0
LLM: 0.111


In [167]:
for cid, score in results[:5]:

    candidate = next(
        c for c in candidates[:1000]
        if c["candidate_id"] == cid
    )

    explanation = explain_candidate(
        candidate,
        jd,
        required_skills,
        required_exp
    )

    print_explanation(explanation)

Candidate:
AI Research Engineer | Data Science & ML enthusiast

Experience:
6.3 years

Matched Skills:
✓ Python

Missing Skills:
✗ LLMs
✗ RAG

Scores:
Semantic: 0.58
Skill: 0.333
Experience: 1.0
LLM: 0.111
Candidate:
Backend Engineer | SQL, Spark, Cloud

Experience:
6.9 years

Matched Skills:

Missing Skills:
✗ LLMs
✗ RAG
✗ Python

Scores:
Semantic: 0.481
Skill: 0.333
Experience: 1.0
LLM: 0.333
Candidate:
AI Research Engineer | Building ML-powered solutions

Experience:
3.6 years

Matched Skills:
✓ RAG

Missing Skills:
✗ LLMs
✗ Python

Scores:
Semantic: 0.558
Skill: 0.333
Experience: 0.5
LLM: 0.333
Candidate:
Sales Executive | AI enthusiast | Building with LLMs

Experience:
4.7 years

Matched Skills:
✓ LLMs

Missing Skills:
✗ RAG
✗ Python

Scores:
Semantic: 0.51
Skill: 0.667
Experience: 0.8
LLM: 0.667
Candidate:
Civil Engineer | AI enthusiast | Building with LLMs

Experience:
11.1 years

Matched Skills:
✓ LLMs
✓ RAG

Missing Skills:
✗ Python

Scores:
Semantic: 0.475
Skill: 1.0
Experien

In [171]:
def build_candidate_embeddings(candidates):

    candidate_embeddings = {}

    for candidate in candidates:

        candidate_embeddings[
            candidate["candidate_id"]
        ] = model.encode(
            build_candidate_profile(candidate)
        )

    return candidate_embeddings

In [173]:
def initialize_embeddings(candidates):

    global candidate_embeddings

    candidate_embeddings = build_candidate_embeddings(
        candidates
    )

In [174]:
extract_skills_from_jd
extract_required_experience
get_candidate_skills
skill_match_score
proficiency_to_score
skill_match_score_with_proficiency
experience_score
candidate_quality_score
career_relevance_score
role_alignment_score
llm_bonus
build_candidate_profile
build_candidate_embeddings
initialize_embeddings
semantic_similarity_cached
final_candidate_score
rank_candidates_semantic
explain_candidate
print_explanation

<function __main__.print_explanation(explanation)>

In [175]:
functions = [
    extract_skills_from_jd,
    extract_required_experience,
    get_candidate_skills,
    skill_match_score,
    proficiency_to_score,
    skill_match_score_with_proficiency,
    experience_score,
    candidate_quality_score,
    career_relevance_score,
    role_alignment_score,
    llm_bonus,
    build_candidate_profile,
    semantic_similarity_cached,
    final_candidate_score,
    rank_candidates_semantic,
    explain_candidate,
    print_explanation
]

print("Total functions:", len(functions))

Total functions: 17
